In [ ]:
from langgraph.graph import StateGraph
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from duckduckgo_search import DDGS
from typing import TypedDict

from pathlib import Path
from decouple import AutoConfig

In [36]:
cwd = Path.cwd()
project_dir = cwd / "langgraph" if (cwd / "langgraph").is_dir() else cwd
config = AutoConfig(search_path=str(project_dir))
api_key = config("OPENAI_API_KEY")
llm = ChatOpenAI(
    api_key=api_key,
    model_name='gpt-6-luna',
    store=False,
    # temperature=0.3
)

Estado

In [37]:
class State(TypedDict):
    pergunta: str
    conteudo: str
    resposta: str

Funçōes de callback

In [38]:
def recebe_pergunta(state: State) -> State:
    print(f"Usuário perguntou: {state['pergunta']}")
    return {"pergunta": state['pergunta']}

In [39]:
def precisa_pesquisar(state: State) -> State:
    pergunta = state['pergunta'].lower()
    precisa = any(p in pergunta for p in ['dados', 'estatística', 'números', 'pesquisa'])
    print(f"Precisa pesquisar? {precisa}")
    return {"next_step": "pesquisar" if precisa else "consultar_llm"}

In [40]:
def pesquisar(state: State) -> State:
    pergunta = state['pergunta']
    print(f"Pesquisando no DuckDuckGo: {pergunta}")

    with DDGS() as ddgs:
        resultados = ddgs.text(pergunta, max_results=1)

    if resultados:
        contexto = "\n".join([r['body'] for r in resultados if 'body' in r])
    else:
        contexto = "Nenhum resultado encontrado."

    return {"conteudo": contexto}

In [41]:
def consultar_llm(state: State) -> State:
    prompt = ChatPromptTemplate.from_template(
        "Responda à seguinte pergunta: {pergunta}"
    )
    chain = prompt | llm
    resposta = chain.invoke({"pergunta": state["pergunta"]})
    print("Resposta direto do LLM.")
    return {"resposta": resposta.content}

In [42]:
def sintetizar(state: State) -> State:
    contexto = state.get("conteudo", "")
    pergunta = state["pergunta"]
    prompt = ChatPromptTemplate.from_template(
        """
        Use o seguinte contexto para responder a pergunta:
        Contexto: {contexto}
        Pergunta: {pergunta}
        Resposta:
        """
    )
    chain = prompt | llm
    resposta = chain.invoke(
        {"contexto": contexto, "pergunta": pergunta}
    )
    print("Resposta sintetizada com contexto.")
    return {"resposta": resposta.content}

In [43]:
def responder(state: State) -> State:
    print("\n Resposta Final:")
    print(state["resposta"])
    return state

Cria Grafo

In [44]:
graph = StateGraph(State)

graph.add_node("recebe_pergunta", RunnableLambda(recebe_pergunta))
graph.add_node("decisao", RunnableLambda(precisa_pesquisar))
graph.add_node("pesquisar", RunnableLambda(pesquisar))
graph.add_node("consultar_llm", RunnableLambda(consultar_llm))
graph.add_node("sintetizar", RunnableLambda(sintetizar))
graph.add_node("responder", RunnableLambda(responder))

Transiçōes de Estado e Condiçōes

In [45]:
graph.set_entry_point("recebe_pergunta")

graph.add_edge("recebe_pergunta", "decisao")
graph.add_conditional_edges(
    "decisao",
    lambda state: state["next_step"],
    {
        "pesquisar": "pesquisar",
        "consultar_llm": "consultar_llm"
    }
)
graph.add_edge("pesquisar", "sintetizar")
graph.add_edge("consultar_llm", "responder")
graph.add_edge("sintetizar", "responder")
graph.set_finish_point("responder")

Execuçāo

In [46]:
executable = graph.compile()

print("\n TESTE 1:")
executable.invoke({"pergunta": "Qual é a Capital da Alemanha?"})

print("\n TESTE 2:")
executable.invoke({"pergunta": "Me mostre dados sobre a economia brasileira em 2026."})


 TESTE 1:
Usuário perguntou: Qual é a Capital da Alemanha?
Precisa pesquisar? False
Resposta direto do LLM.

 Resposta Final:
A capital da Alemanha é Berlim.

 TESTE 2:
Usuário perguntou: Me mostre dados sobre a economia brasileira em 2026.
Precisa pesquisar? True
Pesquisando no DuckDuckGo: Me mostre dados sobre a economia brasileira em 2026.


/var/folders/f5/jv7k5yw97479jf4xr73rgvd40000gn/T/ipykernel_3653/3442949834.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Resposta sintetizada com contexto.

 Resposta Final:
O contexto fornecido não contém dados numéricos sobre a economia brasileira. Como 2026 ainda está em andamento, também não há resultado fechado para o ano inteiro. Para não inventar valores, segue onde consultar os principais indicadores:

| Indicador | Fonte oficial |
|---|---|
| PIB e atividade econômica | [IBGE — Contas Nacionais](https://www.ibge.gov.br/) |
| Inflação (IPCA) | [IBGE — IPCA](https://www.ibge.gov.br/estatisticas/economicas/precos-e-custos.html) |
| Juros (Selic) e projeções econômicas | [Banco Central — Estatísticas](https://www.bcb.gov.br/estatisticas) |
| Desemprego | [IBGE — PNAD Contínua](https://www.ibge.gov.br/estatisticas/sociais/trabalho.html) |
| Emprego formal | [Ministério do Trabalho — Novo Caged](https://www.gov.br/trabalho-e-emprego/) |
| Comércio exterior | [MDIC — Comex Stat](https://comexstat.mdic.gov.br/) |

Os números publicados ao longo do ano são parciais; projeções, como as do Boletim Focus, d

{'pergunta': 'Me mostre dados sobre a economia brasileira em 2026.',
 'conteudo': 'Nenhum resultado encontrado.',
 'resposta': 'O contexto fornecido não contém dados numéricos sobre a economia brasileira. Como 2026 ainda está em andamento, também não há resultado fechado para o ano inteiro. Para não inventar valores, segue onde consultar os principais indicadores:\n\n| Indicador | Fonte oficial |\n|---|---|\n| PIB e atividade econômica | [IBGE — Contas Nacionais](https://www.ibge.gov.br/) |\n| Inflação (IPCA) | [IBGE — IPCA](https://www.ibge.gov.br/estatisticas/economicas/precos-e-custos.html) |\n| Juros (Selic) e projeções econômicas | [Banco Central — Estatísticas](https://www.bcb.gov.br/estatisticas) |\n| Desemprego | [IBGE — PNAD Contínua](https://www.ibge.gov.br/estatisticas/sociais/trabalho.html) |\n| Emprego formal | [Ministério do Trabalho — Novo Caged](https://www.gov.br/trabalho-e-emprego/) |\n| Comércio exterior | [MDIC — Comex Stat](https://comexstat.mdic.gov.br/) |\n\nOs n

Gráfico

In [47]:
dot = executable.get_graph().draw_mermaid()

print(dot)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	recebe_pergunta(recebe_pergunta)
	decisao(decisao)
	pesquisar(pesquisar)
	consultar_llm(consultar_llm)
	sintetizar(sintetizar)
	responder(responder)
	__end__([<p>__end__</p>]):::last
	__start__ --> recebe_pergunta;
	consultar_llm --> responder;
	decisao -.-> consultar_llm;
	decisao -.-> pesquisar;
	pesquisar --> sintetizar;
	recebe_pergunta --> decisao;
	sintetizar --> responder;
	responder --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

